# Pipeline de Pré-processamento e Preparação de Dados

Este notebook consolida a limpeza de dados iniciada na EDA (Issue #4) e prepara os conjuntos de treino, validação e teste para o treinamento dos modelos de classificação (Issue #5).

## Objetivos:
1. Realizar o tratamento de valores nulos.
2. Converter o alvo categórico para numérico.
3. Escalonar os atributos físicos.
4. Dividir os dados em proporção 70/15/15.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configurações
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')

## 1. Carga dos Dados
Utilizaremos a URL oficial para garantir a consistência com a etapa anterior.

In [ ]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv'
df_raw = pd.read_csv(url)
print(f"Dataset carregado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas")

## 2. Seleção de Atributos e Limpeza Inicial
Identificamos na EDA que algumas colunas são totalmente nulas ou são apenas IDs/metadados que não contribuem para a física do problema.

In [ ]:
# Definindo o alvo
target = 'koi_pdisposition'

# Colunas para remover (IDs e metadados irrelevantes para treinamento)
cols_to_drop = [
    'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 
    'koi_tce_delivname', 'koi_fittype', 'ra_str', 'dec_str'
]

# Identificando colunas com mais de 50% de valores nulos (conforme observado na EDA)
null_threshold = 0.5 * len(df_raw)
high_null_cols = df_raw.columns[df_raw.isnull().sum() > null_threshold].tolist()

total_drop = list(set(cols_to_drop + high_null_cols))
df = df_raw.drop(columns=total_drop)

print(f"Colunas removidas: {len(total_drop)}")
print(f"Novo formato: {df.shape}")

## 3. Encoding e Tratamento de Nulos
Transformaremos o alvo em binário (1 e 0) e aplicaremos a mediana nos valores ausentes dos atributos numéricos.

In [ ]:
# Encoding do alvo
le = LabelEncoder()
df[target] = le.fit_transform(df[target])
print(f"Classes mapeadas: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Imputação pela mediana em colunas numéricas
df_numeric = df.select_dtypes(include=[np.number])
df_final = df_numeric.fillna(df_numeric.median())

print(f"Nulos remanescentes: {df_final.isnull().sum().sum()}")

## 4. Divisão dos Dados (Split)
Dividiremos em 70% Treino, 15% Validação e 15% Teste.

In [ ]:
X = df_final.drop(columns=[target])
y = df_final[target]

# Primeiro split: Treino vs Resto (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Segundo split: Validação vs Teste (50% do Resto = 15% cada)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Treino: {X_train.shape}")
print(f"Validação: {X_val.shape}")
print(f"Teste: {X_test.shape}")

## 5. Escalonamento (Scaling)
Padronização é vital para o bom desempenho da MLP.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Escalonamento concluído com StandardScaler.")

## 6. Treinamento do Modelo Baseline (Issue #6)
Utilizaremos a **Regressão Logística** como nosso modelo base (benchmark).

In [ ]:
# Instanciando o modelo
baseline = LogisticRegression(max_iter=1000, random_state=42)

# Treinamento
baseline.fit(X_train_scaled, y_train)

y_pred_base = baseline.predict(X_val_scaled)
acc_base = accuracy_score(y_val, y_pred_base)

print(f"Modelo Baseline treinado! Acurácia na Validação: {acc_base:.4f}")

## 7. Implementação da Rede Neural MLP v1 (Issue #7)
Nesta etapa, implementamos a primeira versão da MLP com uma arquitetura padrão.

In [ ]:
# Definição da MLP v1
mlp_v1 = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    verbose=False,
    early_stopping=True,
    validation_fraction=0.1
)

# Treinamento
mlp_v1.fit(X_train_scaled, y_train)

print(f"MLP v1 treinada em {mlp_v1.n_iter_} iterações.")

### 8. Diagnóstico de Convergência
Plotagem da curva de erro para validar o aprendizado.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mlp_v1.loss_curve_)
plt.title('Curva de Perda (Loss Curve) - MLP v1')
plt.xlabel('Iterações')
plt.ylabel('Perda (Loss)')
plt.grid(True)
plt.show()

## 9. Comparação de Resultados: Baseline vs MLP v1
Abaixo, consolidamos as métricas para a primeira avaliação comparativa.

In [ ]:
y_pred_mlp = mlp_v1.predict(X_val_scaled)
acc_mlp = accuracy_score(y_val, y_pred_mlp)

print("--- Relatório de Classificação: MLP v1 ---")
print(classification_report(y_val, y_pred_mlp, target_names=le.classes_))

print("\n--- Comparativo de Acurácia ---")
print(f"Baseline (Regressão Logística): {acc_base:.4f}")
print(f"Rede Neural (MLP v1):           {acc_mlp:.4f}")

### Conclusão da Issue #7
A MLP v1 foi implementada com sucesso. Com base nos resultados acima, podemos decidir se a arquitetura atual é suficiente ou se precisamos explorar variações na **Issue #8**.